# Goal

Тестируем  `18d_world_model_05`, а именно:
1) факторизованную `WorldModel`: `ReconstructionModel` (spatial, structure) + `PredictionModel` (temporal, dynamics)
2) отчуждаемый `ReconstructionModel`
3) в `ReconstructionModel` используются раздельные `lv_encoding` (как это было в `18d_world_model_03`)
4) `trapezoid` аннилинг для `pred_loss_coef`. Логика: сначала модель должна структурой овладеть, а потом уже предсказывать
5) добавлен слой трансляции из терминов предсказательной модели в термины реконстракт модели (`p2r_translator`). Логика: пусть предсказалка сосредоточится на более высоких абстракциях, а переход от них в абстракциям струтурным сделает транслятор

# GRID_SEARCH_SPACE

In [ ]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [25]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.comment = None
    HP.general.random_seed = random.randint(0, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    
    HP.dataset.train = [
        'train_dataset:100',
        'train_dataset:101',
        'train_dataset:102',
        'train_dataset:103',
        'train_dataset:104',
        'train_dataset:105',
        'train_dataset:106',
        'train_dataset:107',
        'train_dataset:108',
        'train_dataset:109',
    ]
    HP.dataset.test = 'test_dataset:2'

    HP.model.parent = None
    HP.model.sequence_length = 4
    HP.model.actions_count = 6
    HP.model.ob_shape = (1, 178, 152)
    HP.model.d_model = 256
    HP.model.vision_head = dict(grid=(6,6), features_counts=(16, 32, 64, 128))
    HP.model.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.model.render_head = dict(projector='linear', render_engine=dict(type='layer', layers_count=4))
    
    HP.train.epochs_count = 100
    HP.train.batch_size = 128
    HP.train.optimizer = 'AdamW'
    HP.train.max_grad_norm = 1.0
    HP.train.learn_rate = 'const(0.0005)'
    HP.train.bce_loss_coef = 'const(1.0)'
    HP.train.edge_loss_coef = 'const(1.0)'
    HP.train.recon_loss_coef = 'const(1.0)'
    HP.train.pred_loss_coef = 'trapezoid(a=0, b=1, c=1, a_dur=0.25, b_dur=0.75)'
    
    return HP
# @launchit.stop

# Results
<TBD>

Лучший запуск дал `ssim=0.9774` (18d_world_model_05/opt_04.1/13). Для справки:
- лучший прогон `18d_world_model_02` дал `ssim=0.9833` (18d_world_model_02/opt_01/17).
- лучший прогон `18d_world_model_04` дал `ssim=0.9794` (18d_world_model_04/opt_03.1/49).

Чё-то как-то хуже и хуже =) 0.9833->0.9794->0.9774. Понятно, что транслятор также должен обучаться, и м.б. нужно больше эпох. Тем не менее его вклад под вопросом.

<img src="./img/ssim.png">
<img src="./img/recon_ssim.png">
<img src="./img/pred_ssim.png">

**Выводы**
1) кажется, что нужно сначала сосредоточиться на recon модели. А уж потом упражняться с Prediction.
2) надо сделать так, чтобы recon модель работало лучшее всего!